In [ ]:
# Install required libraries
!pip install -q -U transformers accelerate bitsandbytes datasets pandas langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 60.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configuration
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert DEVICE == "cuda", "GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU"

print(f"Loading tokenizer and model: {MODEL_ID} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading tokenizer and model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B on cuda...


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
pip install langdetect

In [ ]:
import re
import os
import time
import pandas as pd
import langdetect

def extract_ground_truth(solution_str):
    """Extracts the final numerical answer from GSM8K ground truth string."""
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", solution_str)
    return match.group(1).replace(",", "").strip() if match else ""

def parse_numeric_answer(text):
    """Extracts the first standalone scalar number from generated response."""
    if not text:
        return ""
    cleaned = text.replace("$", "").replace(",", "")
    numbers = re.findall(r"-?\d+(?:\.\d+)?", cleaned)
    return numbers[0].strip() if numbers else ""

def calculate_target_language_ratio(text, target_lang_code="en"):
    """Calculates the percentage of sentences detected in the target language."""
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if not sentences:
        return 0.0
    target_count = 0
    for sent in sentences:
        try:
            if langdetect.detect(sent) == target_lang_code:
                target_count += 1
        except:
            pass
    return (target_count / len(sentences)) * 100

In [1]:
from datasets import load_dataset

# Experiment Parameters
OUTPUT_CSV = "gsm8k_english_benchmark_results.csv"
LANGUAGE = "English"
LANG_CODE = "en"
TARGET_BUDGETS = [50, 100, 200, 400, 800]

# Load Dataset
raw_dataset = load_dataset("openai/gsm8k", "main", split="test")

# Resume Mechanism: Load completed sample IDs if output CSV exists
completed_sample_ids = set()
if os.path.exists(OUTPUT_CSV):
    existing_df = pd.read_csv(OUTPUT_CSV)
    if "sample_id" in existing_df.columns:
        completed_sample_ids = set(existing_df["sample_id"].astype(str))
    print(f"Resuming experiment. Found {len(completed_sample_ids)} previously completed samples.")
else:
    print("Starting new experiment run...")

print(f"Total questions in test set: {len(raw_dataset)}")

for idx, item in enumerate(raw_dataset):
    question_id = f"GSM8K_{idx+1}"
    question_text = item["question"]
    ground_truth = extract_ground_truth(item["answer"])

    # Format Chat Template
    messages = [{"role": "user", "content": question_text}]
    base_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(base_prompt, return_tensors="pt").to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    for budget in TARGET_BUDGETS:
        sample_id = f"{question_id}_b{budget}"

        # Skip if already processed in a previous run
        if sample_id in completed_sample_ids:
            continue

        # ---------------------------------------------------------
        # STEP 1: Generate Thinking Trace up to Budget
        # ---------------------------------------------------------
        gen_output = model.generate(
            **inputs,
            max_new_tokens=budget,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        raw_gen_text = tokenizer.decode(gen_output[0][prompt_len:], skip_special_tokens=False)

        if "</think>" in raw_gen_text:
            thinking_trace = raw_gen_text.split("</think>")[0].replace("<think>", "").strip()
            cutoff_status = "Finished_Thinking_Naturally"
        else:
            thinking_trace = raw_gen_text.replace("<think>", "").strip()
            cutoff_status = "Thinking_Cut_Off_By_Budget"

        thinking_tokens = len(tokenizer.encode(thinking_trace))
        thinking_words = len(thinking_trace.split())
        thinking_chars = len(thinking_trace)
        lang_ratio = calculate_target_language_ratio(thinking_trace, target_lang_code=LANG_CODE)

        # ---------------------------------------------------------
        # STEP 2: Forced Immediate Answer Generation
        # ---------------------------------------------------------
        continuation_prompt = f"{base_prompt}<think>\n{thinking_trace}\n</think>\n\nFinal Answer: "
        forced_inputs = tokenizer(continuation_prompt, return_tensors="pt").to(DEVICE)
        forced_prompt_len = forced_inputs["input_ids"].shape[1]

        ans_output = model.generate(
            **forced_inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        raw_ans_text = tokenizer.decode(ans_output[0][forced_prompt_len:], skip_special_tokens=True).strip()
        parsed_answer = parse_numeric_answer(raw_ans_text)
        is_correct = (parsed_answer == ground_truth)

        # Record Result
        record = {
            "question_id": question_id,
            "language": LANGUAGE,
            "model": MODEL_ID.split("/")[-1],
            "sample_id": sample_id,
            "full_thinking_text": thinking_trace,
            "answer": parsed_answer,
            "correct?": is_correct,
            "tokens": thinking_tokens,
            "words": thinking_words,
            "characters": thinking_chars,
            "%_thinking_target_lang": lang_ratio,
            "answered_or_cut_off": cutoff_status
        }

        df_row = pd.DataFrame([record])
        df_row.to_csv(OUTPUT_CSV, mode="a", index=False, header=not os.path.exists(OUTPUT_CSV))
        completed_sample_ids.add(sample_id)

    if (idx + 1) % 10 == 0 or (idx + 1) == len(raw_dataset):
        print(f"Processed {idx + 1}/{len(raw_dataset)} questions...")

print(f"\n✅ Benchmarking completed successfully! All results saved to {OUTPUT_CSV}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

NameError: name 'os' is not defined